# FMI Model Exchange in DPsim: paper results

This notebook presents the numerical evidence for the paper scope: analytical nonlinear FMI-to-Norton stamping, explicit single-pass versus rollback/Newton coupling, the switched-converter cases, consistent loaded-start initialization, independent Simulink references, timestep/runtime sensitivity, and event-aligned multirate logging. It reads completed files under `build-fmi/`; it does not rerun simulations.

**Scope boundary:** the switched MathWorks FMU does not replay deterministically, so its realistic converter cases use the explicit single-pass path. The notebook reports the logged coupling error and internal steps and does not present these runs as converged multirate Newton results.

In [ ]:
from pathlib import Path
import json
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

REPO = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "dpsim-fmi/CMakeLists.txt").is_file()
    ),
    None,
)
if REPO is None:
    raise RuntimeError(
        "Start the notebook kernel in the DPsim repository or one of its subdirectories."
    )
VALIDATION = REPO / "build-fmi/fmi-validation"
PAPER = VALIDATION / "gate6-weak-grid/paper"
GATE3 = VALIDATION / "gate3"

required = [
    PAPER / "fixture_studies.json",
    PAPER / "converter_case_metrics.csv",
    PAPER / "weak_grid_refinement_metrics.csv",
    PAPER / "weak_grid_runtime_metrics.csv",
    PAPER / "matlab_load_step_metrics.csv",
    PAPER / "matlab_voltage_sag_metrics.csv",
]
missing = [str(p.relative_to(REPO)) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "Required validation outputs are missing: " + ", ".join(missing)
    )
print(f"Repository: {REPO}")
print("All paper-result tables are present.")

## 1. Analytical Norton tangent and loose-coupling baseline

The nonlinear fixture compares importer-generated Norton conductance and history terms with the analytical tangent at representative operating points. The tolerance check also reports the work count and Newton convergence. A separate sampled-FMU fixture compares rollback/Newton with a frozen-current single-pass baseline on the same network and timestep. Its ideal voltage boundary isolates the coupling formulation; weak-grid timestep sensitivity is shown later.

In [ ]:
fixture = json.loads((PAPER / "fixture_studies.json").read_text())
nonlinear = pd.DataFrame(fixture["nonlinear_sensitivity"]).T
nonlinear.index.name = "setting"
print("Analytical Norton/Jacobian and tolerance check")
display(
    nonlinear.rename(
        columns={
            "max_relative_W_error": "max relative W error",
            "max_relative_history_error": "max relative history error",
            "evaluation_requests": "FMI evaluation requests",
            "evaluation_wall_time_s": "FMI evaluation time [s]",
            "max_newton_iterations": "max Newton iterations",
            "max_scaled_residual": "max scaled residual",
        }
    ).round(8)
)

coupling = pd.DataFrame(fixture["same_network_multirate_coupling"]).T
coupling.index.name = "coupling method"
print("Same sampled FMU/network/timestep: rollback/Newton versus explicit single pass")
display(
    coupling.rename(
        columns={
            "max_current_lag_A": "max terminal-current lag [A]",
            "max_poststep_kcl_defect_A": "max post-step KCL defect [A]",
            "independent_state_error_bound": "state recurrence error bound",
        }
    ).round(9)
)

## 2. Switched-converter benchmark and multirate logging

The three 20 ms cases use a 50 µs network macro step and 1 µs internal FMI events (50 internal steps per macro step). The per-case table reports PCC voltage, converter current, active/reactive power, coupling lag, network passes, and logged internal-step counts. The coupling lag is a diagnostic of the explicit path, not a Newton residual.

In [ ]:
case_metrics = pd.read_csv(PAPER / "converter_case_metrics.csv")
display(
    case_metrics.rename(
        columns={
            "case": "case",
            "window": "measurement window",
            "samples": "samples",
            "pcc_voltage_rms_V": "PCC voltage RMS [V]",
            "converter_current_rms_A": "converter current RMS [A]",
            "mean_P_W": "mean P [W]",
            "mean_Q_var": "mean Q [var]",
            "max_coupling_lag_A": "max coupling lag [A]",
            "max_network_passes": "network passes / macro step",
            "internal_steps_per_macro": "internal FMI steps / macro step",
        }
    ).round(5)
)

# The log field is shown directly, not inferred from the requested timestep.
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
for ax, name in zip(axes, ("loaded_start", "weak_grid_sag", "unbalanced_sag")):
    d = pd.read_csv(GATE3 / name / "logs/fmi_mixed.csv", skipinitialspace=True)
    d.columns = d.columns.str.strip()
    ax.plot(
        d.time * 1e3,
        d.fmi_coupling_error_A,
        color="tab:red",
        lw=0.8,
        label="logged coupling error [A]",
    )
    ax.set_ylabel("lag [A]")
    ax.set_title(name.replace("_", " ").title())
    ax.grid(alpha=0.25)
    ax.legend(loc="upper right")
    # Every macro step records how many internal FMI steps actually executed.
    ax2 = ax.twinx()
    ax2.step(
        d.time * 1e3,
        d.fmi_internal_steps,
        where="post",
        color="tab:blue",
        alpha=0.45,
        lw=0.65,
        label="logged internal steps",
    )
    ax2.set_ylabel("FMI steps / macro")
    ax2.set_ylim(0, max(55, float(d.fmi_internal_steps.max()) + 2))
axes[-1].set_xlabel("Time [ms]")
fig.suptitle(
    "Single-pass multirate diagnostics: each 50 µs macro step advances through 1 µs FMI events"
)
fig.tight_layout()
plt.show()

log_rows = []
for name in ("loaded_start", "weak_grid_sag", "unbalanced_sag"):
    d = pd.read_csv(GATE3 / name / "logs/fmi_mixed.csv", skipinitialspace=True)
    d.columns = d.columns.str.strip()
    log_rows.append(
        {
            "case": name,
            "logged macro samples": len(d),
            "internal FMI steps min": int(d.fmi_internal_steps.min()),
            "internal FMI steps max": int(d.fmi_internal_steps.max()),
            "network passes max": int(d.newton_iterations.max()),
            "max reported residual (not converged in single pass)": float(
                d.newton_residual.max()
            ),
            "coupling lag max [A]": float(d.fmi_coupling_error_A.max()),
            "coupling lag max after 1 ms [A]": float(
                d.loc[d.time >= 0.001, "fmi_coupling_error_A"].max()
            ),
        }
    )
display(pd.DataFrame(log_rows).round(6))

## 2a. Consistent loaded-start initialization

The dedicated 50 µs loaded-start run initializes the source, PCC, passive R-L line, native load, and FMU terminal voltage before the t=0 sample is logged. Its phasor initialization accounts for line impedance and load; the line current balances the native load and the FMU start current. The same initialization was checked in the loaded-start, balanced weak-grid sag, and unbalanced sag cases. The later lag trace remains the explicit single-pass coupling diagnostic. The FMU continuous states remain the exported FMI start values, so this is a phasor-consistent network initialization rather than a periodic steady-state solve for the converter controller.

In [ ]:
INIT_CASES = VALIDATION / "consistent-initialization/cases"
GATE3 = VALIDATION / "gate3"


def read_case(folder):
    frame = pd.read_csv(folder / "logs/fmi_mixed.csv", skipinitialspace=True)
    frame.columns = frame.columns.str.strip()
    return frame


def vec(frame, name):
    return frame[[f"{name}_{k}" for k in range(3)]].to_numpy(dtype=float)


case_windows = {
    "loaded_start": (0.010, 0.020),
    "weak_grid_sag": (0.005, 0.010),
    "unbalanced_sag": (0.005, 0.010),
}
rows = []
for name, (start, end) in case_windows.items():
    current = read_case(INIT_CASES / name)
    previous = read_case(GATE3 / name)
    kcl = (
        vec(current, "line_current")
        + vec(current, "load_current")
        + vec(current, "converter_current")
    )
    source_pcc_drop = np.max(
        np.abs(vec(current, "grid_voltage")[0] - vec(current, "pcc_voltage")[0])
    )
    mask = (current.time >= start - 1e-12) & (
        current.time <= end + 1e-12
        if name == "loaded_start"
        else current.time < end - 1e-12
    )
    rows.append(
        {
            "case": name,
            "initial KCL residual [A]": np.max(np.abs(kcl[0])),
            "initial grid-to-PCC drop [V]": source_pcc_drop,
            "paper-window PCC RMS [V]": np.sqrt(
                np.mean(vec(current, "pcc_voltage")[mask] ** 2)
            ),
            "paper-window PCC max change vs previous [V]": np.max(
                np.abs(
                    vec(current, "pcc_voltage")[mask]
                    - vec(previous, "pcc_voltage")[mask]
                )
            ),
            "paper-window current max change vs previous [A]": np.max(
                np.abs(
                    vec(current, "converter_current")[mask]
                    - vec(previous, "converter_current")[mask]
                )
            ),
        }
    )
case_init_metrics = pd.DataFrame(rows)
display(case_init_metrics.round(6))

init = read_case(INIT_CASES / "loaded_start")
old_init = read_case(GATE3 / "loaded_start")
source0 = vec(init, "grid_voltage")[0]
old_kcl_implied = (
    vec(old_init, "line_current")[0]
    + vec(old_init, "converter_current")[0]
    + vec(old_init, "pcc_voltage")[0] / 12500.0
)
new_kcl = (
    vec(init, "line_current")
    + vec(init, "load_current")
    + vec(init, "converter_current")
)
init_compare = pd.DataFrame(
    [
        {
            "initialization": "previous",
            "max |PCC − source| [V]": np.max(
                np.abs(vec(old_init, "pcc_voltage")[0] - source0)
            ),
            "implied t=0 KCL residual [A]": np.max(np.abs(old_kcl_implied)),
            "logged t=0 branch current max [A]": max(
                np.max(np.abs(vec(old_init, "line_current")[0])),
                np.max(np.abs(vec(old_init, "load_current")[0])),
            ),
        },
        {
            "initialization": "consistent loaded start",
            "max |PCC − source| [V]": np.max(
                np.abs(vec(init, "pcc_voltage")[0] - source0)
            ),
            "implied t=0 KCL residual [A]": np.max(np.abs(new_kcl[0])),
            "logged t=0 branch current max [A]": max(
                np.max(np.abs(vec(init, "line_current")[0])),
                np.max(np.abs(vec(init, "load_current")[0])),
            ),
        },
    ]
)
display(init_compare.round(6))
print(
    "Loaded-start explicit lag in first 1 ms:",
    float(init.loc[init.time < 0.001, "fmi_coupling_error_A"].max()),
    "A",
)
print(
    "Loaded-start explicit lag after 1 ms:",
    float(init.loc[init.time >= 0.001, "fmi_coupling_error_A"].max()),
    "A",
)

window = init.time <= 0.001
fig, axes = plt.subplots(3, 1, figsize=(9, 7), sharex=True)
t_ms = init.loc[window, "time"].to_numpy() * 1e3
axes[0].plot(t_ms, init.loc[window, "grid_voltage_1"], label="source phase B")
axes[0].plot(t_ms, init.loc[window, "pcc_voltage_1"], label="PCC phase B")
axes[0].set_ylabel("Voltage [V]")
axes[0].legend(ncol=2)
axes[1].plot(t_ms, init.loc[window, "line_current_1"], label="line")
axes[1].plot(t_ms, init.loc[window, "load_current_1"], label="load")
axes[1].plot(t_ms, init.loc[window, "converter_current_1"], label="FMU")
axes[1].set_ylabel("Current [A]")
axes[1].legend(ncol=3)
axes[2].plot(t_ms, init.loc[window, "fmi_coupling_error_A"], color="tab:red")
axes[2].set_ylabel("Coupling lag [A]")
axes[2].set_xlabel("Time [ms]")
for ax in axes:
    ax.grid(alpha=0.25)
fig.suptitle("Loaded-start initialization and first-step explicit coupling")
fig.tight_layout()
plt.show()

## 3. Independent MATLAB/Simulink physical references

The saved physical models cover matching 0.2 s balanced-load, unbalanced-load, and balanced voltage-sag cases. Their breaker closes at 150 ms and the load/grid event occurs at 170 ms. These traces are independent physical references for the corresponding long DPsim runs; they are not direct pointwise references for the separate 20 ms weak-grid/unbalanced-voltage-sag cases above. The reported windows isolate the unenergized period, breaker startup, pre-load interval, and post-event response.

In [ ]:
load_metrics = pd.read_csv(PAPER / "matlab_load_step_metrics.csv")
selected = load_metrics[
    load_metrics.window.isin(
        [
            "unenergized_0_150ms",
            "breaker_startup_150_160ms",
            "pre_load_step_160_170ms",
            "load_step_170_1999ms",
        ]
    )
    & load_metrics.signal.isin(
        [
            "pcc_voltage_a",
            "converter_current_a",
            "instantaneous_active_power",
            "instantaneous_reactive_power",
            "step_load_current_a",
        ]
    )
].copy()
display(
    selected[
        [
            "scenario",
            "window",
            "signal",
            "max_abs_error",
            "rms_error",
            "max_error_percent_base",
            "rms_error_percent_base",
        ]
    ].round(6)
)

sag_metrics = pd.read_csv(PAPER / "matlab_voltage_sag_metrics.csv")
sag_rows = sag_metrics[
    (sag_metrics.window == "sag_170_190ms")
    & sag_metrics.signal.isin(["v_pcc_a", "i_converter_a", "pll_frequency", "pi_q"])
]
print("Balanced voltage sag, 170–190 ms")
display(
    sag_rows[
        [
            "signal",
            "max_abs_error",
            "rms_error",
            "max_error_percent_base",
            "rms_error_percent_base",
        ]
    ].round(6)
)

for filename, caption in [
    ("matlab_load_step_overlay.png", "Balanced and unbalanced load-step overlays"),
    ("matlab_voltage_sag_overlay.png", "Balanced voltage-sag overlay"),
    ("matlab_voltage_sag_errors.png", "Voltage-sag error traces"),
]:
    print(caption)
    display(Image(filename=str(PAPER / filename)))

## 4. Timestep refinement and runtime/evaluation counters

The 1 µs DPsim run is the fine-step numerical reference for the weak-grid case. The 25 and 50 µs runs use the explicit event-aligned path with 25 and 50 internal steps, respectively. The `newton_residual` field in those single-pass runs is an uniterated implicit mismatch and must not be read as a converged Newton residual.

In [ ]:
refinement = pd.read_csv(PAPER / "weak_grid_refinement_metrics.csv")
ref_rows = refinement[
    refinement.window.eq("sag_5_10ms")
    & refinement.signal.isin(
        ["pcc_voltage_0", "converter_current_0", "pi_q", "P_W", "Q_var"]
    )
    & refinement.step_us.isin([25, 50])
]
display(
    ref_rows.pivot(
        index=["step_us", "signal"],
        columns="window",
        values=["max_abs_error", "rms_error"],
    ).round(5)
)

runtime = pd.read_csv(PAPER / "weak_grid_runtime_metrics.csv")
display(
    runtime.rename(
        columns={
            "step_us": "network step [µs]",
            "wall_time_s": "process time [s]",
            "fmi_evaluation_requests": "FMI evaluation requests",
            "fmi_evaluation_wall_time_s": "FMI evaluation time [s]",
            "max_newton_iterations": "max Newton iterations",
            "max_reported_residual": "max reported residual (mode dependent)",
            "max_coupling_error_A_after_1ms": "max lag after 1 ms [A]",
            "internal_steps_min": "internal steps min",
            "internal_steps_max": "internal steps max",
        }
    ).round(6)
)

for filename, caption in [
    ("weak_grid_refinement_waveforms.png", "Weak-grid waveforms at 1, 25, and 50 µs"),
    (
        "weak_grid_refinement_errors.png",
        "Coarse-step error against the 1 µs DPsim reference",
    ),
]:
    print(caption)
    display(Image(filename=str(PAPER / filename)))

## 5. Multirate rollback result and paper scope

The synthetic sampled fixture validates deterministic event-aligned rollback and Newton coupling. The switched MathWorks R2025a export does not support deterministic replay: identical rollback changes the hidden switched trajectory, and its serialized-state path aborts in the exported binary. Consequently the realistic converter evidence in this notebook is for the explicit single-pass path, whose actual per-step internal counts and coupling lag are plotted above. Do not claim realistic multirate Newton convergence from these cases.

**Supported core claim:** a generic nonlinear FMI Model Exchange component can be condensed to a Norton tangent and iterated within DPsim’s EMT MNA/Newton solve. The switched converter demonstrates event-aligned single-pass operation and quantified coupling error. If the paper retains “multirate” in its title, it must describe this explicit limitation precisely.

## Reproduction

This notebook reads existing result files only. To regenerate the three long MATLAB/Simulink references, run `dpsim-fmi/examples/inverter/matlab/validation/regenerate_physical_references.m` in MATLAB. Then run `python3 dpsim-fmi/examples/inverter/run_paper_validation.py --reuse` from the repository root to regenerate the metrics and figures. The underlying DPsim configurations and CSV logs are under `build-fmi/fmi-validation/`.

To regenerate the full-resolution consistent-initialization run, build `FMI_EMT_Ph3_Mixed`, create `build-fmi/fmi-validation/consistent-initialization/`, and run the executable there with `dpsim-fmi/examples/inverter/sources/GridTiedInverterPortedV3.x86_64-linux.fmu` and `dpsim-fmi/examples/inverter/sources/consistent_loaded_start_50us.json`. The CSV is written to that directory's `logs/fmi_mixed.csv`.